# Amazon Review Moat Analysis — AMZ Data Intelligence

**Date**: 2026-06-02  
**Analyst**: Data Storyteller  
**Data**: 984 unique products across 10 Amazon categories (Best Sellers, scraped 2026-06-01)

## Key Findings
- **Only 0.8%** of top-100 Amazon products have fewer than 50 reviews — the "Review Moat" is real
- **But 3/10 categories are "Review-Independent"**: Electronics, Sports & Outdoors, and Toys & Games — review count does NOT significantly predict BSR rank
- **Toys & Games**: Star rating matters MORE than review count for predicting rank (the only category where quality beats quantity)
- **"Two-Tier Amazon"**: Categories fall into Review-Locked (7/10) vs Review-Independent (3/10) types
- **Category Accessibility Index**: Toys & Games (73.9/100) is easiest to enter; Baby (0.2/100) is nearly impenetrable

## Contents
This notebook reproduces the complete analysis: EDA, Spearman correlations, logistic regression, Mann-Whitney U tests, Kruskal-Wallis, Category Accessibility Index, and 8 visualizations.

See `../tasks/active/review-moat-analysis/analysis.md` for the full written analysis with Reddit/LinkedIn/GitHub content drafts.

In [1]:
"""
Review Moat Analysis — Phase B: Data Science Execution
Data Storyteller, AMZ Data Intelligence
Date: 2026-06-02
"""
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from scipy import stats
from scipy.stats import spearmanr, mannwhitneyu, kruskal
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
import warnings, os, sys
warnings.filterwarnings('ignore')

# ── Setup ──────────────────────────────────────────────
DATA_PATH = "../../data/raw/2026-06-01_amazon_best_sellers_review-moat.csv"
VIZ_DIR  = "../../data/visualizations/review-moat-analysis"
DATE     = "2026-06-02"
os.makedirs(VIZ_DIR, exist_ok=True)

# Color palette (colorblind-friendly via seaborn 'colorblind')
sns.set_theme(style="whitegrid", palette="colorblind", font="sans-serif")
CB_PALETTE = sns.color_palette("colorblind", 10)
plt.rcParams.update({'figure.dpi': 150, 'savefig.dpi': 150, 'savefig.bbox': 'tight'})


In [2]:
# ── Load & Clean ───────────────────────────────────────


In [3]:
print("=" * 70)
print("1. LOADING DATA")


1. LOADING DATA


In [4]:
print("=" * 70)

df = pd.read_csv(DATA_PATH)
print(f"Raw shape: {df.shape}")
print(f"Columns: {list(df.columns)}")

# Deduplicate by ASIN (keep lowest BSR rank = best rank)
df = df.sort_values('bsr_rank').drop_duplicates(subset='asin', keep='first').reset_index(drop=True)

# Convert price from JPY to USD (approximate: 1 JPY ≈ 0.0069 USD as of mid-2026)
JPY_TO_USD = 0.0069
df['price_usd'] = df['price'] * JPY_TO_USD

# Create derived columns
df['log_reviews'] = np.log1p(df['review_count'])
df['in_top20'] = (df['bsr_rank'] <= 20).astype(int)
df['rank_tier'] = pd.cut(df['bsr_rank'], bins=[0, 20, 50, 100], labels=['Top 20', '21-50', '51-100'])
df['price_usd'] = df['price_usd'].fillna(df.groupby('category')['price_usd'].transform('median'))
df['rating'] = df['rating'].fillna(df.groupby('category')['rating'].transform('median'))

print(f"After dedup: {df.shape[0]} rows")
print(f"Categories: {df['category'].nunique()}")
print(f"Missing after fill — price_usd: {df['price_usd'].isna().sum()}, rating: {df['rating'].isna().sum()}")



Raw shape: (996, 10)
Columns: ['category', 'bsr_rank', 'asin', 'title', 'rating', 'review_count', 'price', 'price_raw', 'is_sponsored', 'scraped_at']
After dedup: 984 rows
Categories: 10
Missing after fill — price_usd: 0, rating: 0


In [5]:
# ── EDA ────────────────────────────────────────────────


In [6]:
print("\n" + "=" * 70)
print("2. EXPLORATORY DATA ANALYSIS")



2. EXPLORATORY DATA ANALYSIS


In [7]:
print("=" * 70)

print("\n── Per-Category Summary ──")
cat_summary = df.groupby('category').agg(
    n_products=('asin', 'count'),
    median_bsr=('bsr_rank', 'median'),
    mean_reviews=('review_count', 'mean'),
    median_reviews=('review_count', 'median'),
    mean_rating=('rating', 'mean'),
    mean_price_usd=('price_usd', 'mean'),
    median_price_usd=('price_usd', 'median'),
    pct_top20_low_reviews=('in_top20', lambda x: (df.loc[x.index, 'review_count'] <= 50).sum() / len(x) * 100),
    products_under_50_reviews=('review_count', lambda x: (x <= 50).sum()),
    products_under_100_reviews=('review_count', lambda x: (x <= 100).sum()),
    products_under_500_reviews=('review_count', lambda x: (x <= 500).sum()),
).round(2)

cat_summary['pct_under_50'] = (cat_summary['products_under_50_reviews'] / cat_summary['n_products'] * 100).round(1)
cat_summary['pct_under_100'] = (cat_summary['products_under_100_reviews'] / cat_summary['n_products'] * 100).round(1)
cat_summary['pct_under_500'] = (cat_summary['products_under_500_reviews'] / cat_summary['n_products'] * 100).round(1)
print(cat_summary.to_string())

print("\n── Review Count Distribution ──")
print(df['review_count'].describe())
print(f"\n% with < 50 reviews: {(df['review_count'] < 50).mean()*100:.1f}%")
print(f"% with < 100 reviews: {(df['review_count'] < 100).mean()*100:.1f}%")
print(f"% with < 500 reviews: {(df['review_count'] < 500).mean()*100:.1f}%")
print(f"Max reviews: {df['review_count'].max():,}")

print("\n── Rank Tier Distribution ──")
print(df['rank_tier'].value_counts())
print(f"\nTop 20 breakdown:")
print(df[df['in_top20'] == 1].groupby('category').size())



── Per-Category Summary ──
                          n_products  median_bsr  mean_reviews  median_reviews  mean_rating  mean_price_usd  median_price_usd  pct_top20_low_reviews  products_under_50_reviews  products_under_100_reviews  products_under_500_reviews  pct_under_50  pct_under_100  pct_under_500
category                                                                                                                                                                                                                                                                           
Baby                             100        50.5      33906.54         25506.5         4.65           25.71             21.92                   0.00                          0                           0                           1           0.0            0.0            1.0
Electronics                       99        51.0      35363.34         19015.0         4.54           82.43             27.10                   

In [8]:
# ── STATISTICAL TESTS ──────────────────────────────────


In [9]:
print("\n" + "=" * 70)
print("3. STATISTICAL TESTS")



3. STATISTICAL TESTS


In [10]:
print("=" * 70)

results = {}  # store per-category results

for cat in sorted(df['category'].unique()):
    cat_df = df[df['category'] == cat]
    print(f"\n── {cat} (n={len(cat_df)}) ──")

    # Spearman rank correlation
    rho, p_spearman = spearmanr(cat_df['review_count'], cat_df['bsr_rank'])
    print(f"  Spearman ρ(reviews, BSR): {rho:.3f}, p={p_spearman:.4f} {'***' if p_spearman < 0.001 else '**' if p_spearman < 0.01 else '*' if p_spearman < 0.05 else 'ns'}")

    # Logistic regression: P(top-20) ~ log(reviews) + price + rating
    X = cat_df[['log_reviews', 'price_usd', 'rating']].values
    y = cat_df['in_top20'].values
    lr = LogisticRegression(max_iter=1000)
    try:
        lr.fit(X, y)
        # Review count at which P(top-20) = 0.5
        # logit(p) = b0 + b1*log_reviews + b2*price + b3*rating
        # log_reviews_for_p50 = -(b0 + b2*median_price + b3*median_rating) / b1
        b0, b1, b2, b3 = lr.intercept_[0], lr.coef_[0][0], lr.coef_[0][1], lr.coef_[0][2]
        median_price = cat_df['price_usd'].median()
        median_rating = cat_df['rating'].median()
        log_rev_threshold = -(b0 + b2*median_price + b3*median_rating) / b1
        review_threshold = np.exp(log_rev_threshold) - 1  # reverse log1p
        review_threshold = max(0, review_threshold)
        print(f"  Logistic: P(top-20)=0.5 at ~{review_threshold:.0f} reviews (log_reviews coef={b1:.3f})")
    except Exception as e:
        b1, review_threshold = np.nan, np.nan
        print(f"  Logistic regression failed: {e}")

    # Probability by review bucket
    bins = [0, 10, 25, 50, 100, 250, 500, 1000, float('inf')]
    labels = ['0-10', '11-25', '26-50', '51-100', '101-250', '251-500', '501-1000', '1000+']
    cat_df['review_bucket'] = pd.cut(cat_df['review_count'], bins=bins, labels=labels)
    bucket_prob = cat_df.groupby('review_bucket', observed=False)['in_top20'].agg(['mean', 'count'])

    results[cat] = {
        'n': len(cat_df),
        'spearman_rho': rho,
        'spearman_p': p_spearman,
        'review_threshold': review_threshold,
        'log_reviews_coef': b1,
        'median_reviews': cat_df['review_count'].median(),
        'pct_under_50': (cat_df['review_count'] <= 50).mean() * 100,
        'pct_under_100': (cat_df['review_count'] <= 100).mean() * 100,
        'median_price_usd': cat_df['price_usd'].median(),
        'price_cv': cat_df['price_usd'].std() / cat_df['price_usd'].mean() if cat_df['price_usd'].mean() > 0 else 0,
        'bucket_probs': bucket_prob,
    }




── Baby (n=100) ──
  Spearman ρ(reviews, BSR): -0.445, p=0.0000 ***
  Logistic: P(top-20)=0.5 at ~110188 reviews (log_reviews coef=1.102)

── Electronics (n=99) ──
  Spearman ρ(reviews, BSR): -0.180, p=0.0752 ns
  Logistic: P(top-20)=0.5 at ~0 reviews (log_reviews coef=-0.018)

── Health & Household (n=100) ──
  Spearman ρ(reviews, BSR): -0.297, p=0.0027 **
  Logistic: P(top-20)=0.5 at ~748774 reviews (log_reviews coef=0.558)

── Home & Kitchen (n=91) ──
  Spearman ρ(reviews, BSR): -0.235, p=0.0247 *
  Logistic: P(top-20)=0.5 at ~508045 reviews (log_reviews coef=0.535)

── Kitchen & Dining (n=100) ──
  Spearman ρ(reviews, BSR): -0.219, p=0.0288 *
  Logistic: P(top-20)=0.5 at ~308745 reviews (log_reviews coef=0.518)

── Office Products (n=100) ──
  Spearman ρ(reviews, BSR): -0.335, p=0.0007 ***
  Logistic: P(top-20)=0.5 at ~106778 reviews (log_reviews coef=1.150)

── Pet Supplies (n=95) ──
  Spearman ρ(reviews, BSR): -0.353, p=0.0004 ***
  Logistic: P(top-20)=0.5 at ~180895 reviews (lo

In [11]:
# ── Cross-category tests ──
print("\n── Cross-Category Tests ──")
# Kruskal-Wallis: does review_count differ across categories?
kw_stat, kw_p = kruskal(*[df[df['category']==c]['review_count'].values for c in sorted(df['category'].unique())])
print(f"Kruskal-Wallis H: {kw_stat:.1f}, p={kw_p:.2e} — categories {'DO' if kw_p < 0.05 else 'do NOT'} differ significantly")

# Mann-Whitney U for rank tiers
top20 = df[df['in_top20']==1]['review_count']
rest = df[df['in_top20']==0]['review_count']
mw_stat, mw_p = mannwhitneyu(top20, rest, alternative='two-sided')
print(f"Mann-Whitney U (top20 vs rest): U={mw_stat:,.0f}, p={mw_p:.2e}")

# Rating vs Reviews comparison per category
print("\n── Standardized Coefficients: Rating vs log(Reviews) predicting BSR ──")
for cat in sorted(df['category'].unique()):
    cat_df = df[df['category'] == cat].copy()
    scaler = StandardScaler()
    X = scaler.fit_transform(cat_df[['log_reviews', 'rating']])
    y = -cat_df['bsr_rank'].values  # negate so higher = better rank
    # Simple OLS-ish: correlation comparison
    r_reviews = spearmanr(cat_df['log_reviews'], cat_df['bsr_rank'])[0]
    r_rating = spearmanr(cat_df['rating'], cat_df['bsr_rank'])[0]
    stronger = "Reviews" if abs(r_reviews) > abs(r_rating) else "Rating"
    print(f"  {cat:<30s}: ρ_reviews={r_reviews:+.3f}  ρ_rating={r_rating:+.3f}  → {stronger} matters more")




── Cross-Category Tests ──
Kruskal-Wallis H: 144.4, p=1.28e-26 — categories DO differ significantly
Mann-Whitney U (top20 vs rest): U=100,570, p=1.04e-10

── Standardized Coefficients: Rating vs log(Reviews) predicting BSR ──
  Baby                          : ρ_reviews=-0.445  ρ_rating=-0.246  → Reviews matters more
  Electronics                   : ρ_reviews=-0.180  ρ_rating=-0.161  → Reviews matters more
  Health & Household            : ρ_reviews=-0.297  ρ_rating=+0.072  → Reviews matters more
  Home & Kitchen                : ρ_reviews=-0.235  ρ_rating=+0.074  → Reviews matters more
  Kitchen & Dining              : ρ_reviews=-0.219  ρ_rating=-0.085  → Reviews matters more
  Office Products               : ρ_reviews=-0.335  ρ_rating=-0.145  → Reviews matters more


  Pet Supplies                  : ρ_reviews=-0.353  ρ_rating=-0.116  → Reviews matters more
  Sports & Outdoors             : ρ_reviews=-0.121  ρ_rating=-0.081  → Reviews matters more
  Tools & Home Improvement      : ρ_reviews=-0.319  ρ_rating=-0.068  → Reviews matters more
  Toys & Games                  : ρ_reviews=-0.062  ρ_rating=+0.106  → Rating matters more


In [12]:
# ── Category Accessibility Index ──
print("\n── Category Accessibility Index ──")
# Component 1: Review barrier (% of top-100 with ≤50 reviews) — higher = better
# Component 2: Price dispersion (CV) — higher = more room to differentiate
# Component 3: Concentration (share of top 20 held by top 3 brands — proxy with title keyword concentration)
# For concentration: % of top 20 products whose title starts with same first word as top 3 brands
def calc_concentration(cat_df):
    """Approximate brand concentration: share of top 20 held by 3 most common title-start words."""
    top20_titles = cat_df[cat_df['in_top20'] == 1]['title']
    first_words = top20_titles.str.split().str[0].value_counts()
    top3_share = first_words.head(3).sum() / len(top20_titles) if len(top20_titles) > 0 else 0.5
    return top3_share

accessibility = []
for cat in sorted(df['category'].unique()):
    cat_df = df[df['category'] == cat]
    pct_under_50 = (cat_df['review_count'] <= 50).mean() * 100
    price_cv = cat_df['price_usd'].std() / cat_df['price_usd'].mean() if cat_df['price_usd'].mean() > 0 else 0
    concentration = calc_concentration(cat_df)
    accessibility.append({
        'category': cat,
        'pct_under_50_reviews': pct_under_50,
        'price_cv': price_cv,
        'brand_concentration_top3': concentration,
    })

acc_df = pd.DataFrame(accessibility)

# Min-max normalize each component to 0-100
for col in ['pct_under_50_reviews', 'price_cv', 'brand_concentration_top3']:
    min_v, max_v = acc_df[col].min(), acc_df[col].max()
    if max_v - min_v > 0:
        acc_df[f'{col}_norm'] = (acc_df[col] - min_v) / (max_v - min_v) * 100
    else:
        acc_df[f'{col}_norm'] = 50

# For brand_concentration: LOWER concentration = BETTER (invert)
acc_df['brand_concentration_top3_norm'] = 100 - acc_df['brand_concentration_top3_norm']

# Weighted composite
acc_df['accessibility_index'] = (
    0.40 * acc_df['pct_under_50_reviews_norm'] +
    0.30 * acc_df['price_cv_norm'] +
    0.30 * acc_df['brand_concentration_top3_norm']
)

acc_df = acc_df.sort_values('accessibility_index', ascending=False)
print(acc_df[['category', 'pct_under_50_reviews', 'price_cv', 'accessibility_index']].to_string(index=False))

print("\n[OK] Statistical analysis complete. Generating visualizations...")




── Category Accessibility Index ──


                category  pct_under_50_reviews  price_cv  accessibility_index
            Toys & Games              3.000000  0.856638            73.941023
             Electronics              3.030303  2.174230            72.727273
        Kitchen & Dining              1.000000  0.808786            38.427316
Tools & Home Improvement              0.000000  1.168452            31.370268
      Health & Household              1.000000  0.633726            29.563636
       Sports & Outdoors              0.000000  0.866986            29.087998
          Home & Kitchen              0.000000  0.955066            26.257822
            Pet Supplies              0.000000  0.683936            22.795979
         Office Products              0.000000  1.371509            17.094964
                    Baby              0.000000  0.643258             0.185619

[OK] Statistical analysis complete. Generating visualizations...


In [13]:
# ── VISUALIZATIONS ─────────────────────────────────────


In [14]:
print("\n" + "=" * 70)
print("4. CREATING VISUALIZATIONS")



4. CREATING VISUALIZATIONS


In [15]:
print("=" * 70)



In [16]:
# --- Chart 1: Review Count vs BSR Rank (facet grid) ---
print("  Chart 1: Review vs BSR Facet Grid")
fig, axes = plt.subplots(2, 5, figsize=(20, 9), sharex=False, sharey=True)
axes = axes.flatten()
for i, (cat, ax) in enumerate(zip(sorted(df['category'].unique()), axes)):
    cat_df = df[df['category'] == cat]
    ax.scatter(cat_df['review_count'], cat_df['bsr_rank'], alpha=0.5, s=20, c=[CB_PALETTE[i]])
    # LOESS-like: simple rolling mean
    sorted_df = cat_df.sort_values('review_count')
    from scipy.interpolate import make_interp_spline
    try:
        x_sorted = sorted_df['review_count'].values
        y_sorted = sorted_df['bsr_rank'].values
        # Use lowess from statsmodels if available, else simple poly
        from statsmodels.nonparametric.smoothers_lowess import lowess
        lowess_result = lowess(y_sorted, x_sorted, frac=0.4, return_sorted=True)
        ax.plot(lowess_result[:, 0], lowess_result[:, 1], 'k-', linewidth=1.5, alpha=0.8)
    except Exception:
        pass
    ax.set_title(cat.replace(' & ', '\n& '), fontsize=9, fontweight='bold')
    ax.set_xlabel('Reviews' if i >= 5 else '')
    ax.set_ylabel('BSR Rank' if i in [0, 5] else '')
    ax.invert_yaxis()
    ax.set_xlim(left=-50)
fig.suptitle('Review Count vs BSR Rank by Category\n(Lower rank = better. Black curve = LOESS trend)', fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
fig.savefig(f"{VIZ_DIR}/{DATE}_chart1_review_vs_bsr_facet.png", bbox_inches='tight', dpi=150)
plt.close()
print("    [OK] Saved")



  Chart 1: Review vs BSR Facet Grid


    [OK] Saved


In [17]:
# --- Chart 2: Review Distribution by Rank Tier (Box Plot) ---
print("  Chart 2: Review Distribution by Rank Tier")
fig, axes = plt.subplots(2, 5, figsize=(20, 9))
axes = axes.flatten()
for i, (cat, ax) in enumerate(zip(sorted(df['category'].unique()), axes)):
    cat_df = df[df['category'] == cat]
    # Filter: only show up to 99th percentile to avoid extreme outliers
    cutoff = cat_df['review_count'].quantile(0.95)
    plot_df = cat_df[cat_df['review_count'] <= cutoff]
    bp = plot_df.boxplot(column='review_count', by='rank_tier', ax=ax, patch_artist=True,
                         boxprops=dict(facecolor=CB_PALETTE[i], alpha=0.6))
    ax.set_title(cat.replace(' & ', '\n& '), fontsize=9, fontweight='bold')
    ax.set_xlabel('')
    ax.set_ylabel('Reviews' if i in [0, 5] else '')
    ax.tick_params(axis='x', rotation=30, labelsize=7)
fig.suptitle('Review Count Distribution by Rank Tier\n(Outliers beyond 95th percentile trimmed for visibility)', fontsize=13, fontweight='bold')
plt.tight_layout()
fig.savefig(f"{VIZ_DIR}/{DATE}_chart2_review_by_tier_boxplot.png", bbox_inches='tight', dpi=150)
plt.close()
print("    [OK] Saved")



  Chart 2: Review Distribution by Rank Tier


    [OK] Saved


In [18]:
# --- Chart 3: Probability of Top-20 by Review Bucket ---
print("  Chart 3: Probability of Top-20 by Review Bucket")
fig, ax = plt.subplots(figsize=(14, 7))
for i, (cat, res) in enumerate(results.items()):
    bp = res['bucket_probs'].reset_index()
    # Only plot buckets with enough data
    bp = bp[bp['count'] >= 3]
    if len(bp) >= 3:
        ax.plot(range(len(bp)), bp['mean'], 'o-', color=CB_PALETTE[i], label=cat, linewidth=1.8, markersize=6)
ax.axhline(y=0.5, color='red', linestyle='--', alpha=0.5, linewidth=1, label='P=0.5 threshold')
ax.set_xticks(range(8))
ax.set_xticklabels(['0-10', '11-25', '26-50', '51-100', '101-250', '251-500', '501-1000', '1000+'])
ax.set_xlabel('Review Count Bucket')
ax.set_ylabel('Probability of Top-20 Rank')
ax.set_title('Probability of Breaking into Top 20 by Review Count\n(The "Review Moat" Curve)', fontsize=14, fontweight='bold')
ax.legend(bbox_to_anchor=(1.02, 1), loc='upper left', fontsize=8, title='Category')
ax.set_ylim(0, 1)
plt.tight_layout()
fig.savefig(f"{VIZ_DIR}/{DATE}_chart3_probability_curve.png", bbox_inches='tight', dpi=150)
plt.close()
print("    [OK] Saved")



  Chart 3: Probability of Top-20 by Review Bucket
    [OK] Saved


In [19]:
# --- Chart 4: The Review Moat — Threshold Summary (Split Type A / Type B) ---
print("  Chart 4: Review Moat Bars (Type A/B split)")

# Classify categories into Type A (review-locked, p < 0.05) and Type B (review-independent, p >= 0.05)
thresholds_list = []
for cat, res in results.items():
    p = res['spearman_p']
    is_significant = p < 0.05
    sig_label = '***' if p < 0.001 else '**' if p < 0.01 else '*' if p < 0.05 else 'ns'
    thresholds_list.append({
        'category': cat,
        'review_moat': max(0, res['review_threshold'] or 0),
        'spearman_p': p,
        'spearman_rho': res['spearman_rho'],
        'significant': is_significant,
        'sig_label': sig_label,
        'type': 'Type A: Review-Locked' if is_significant else 'Type B: Review-Independent'
    })

threshold_df = pd.DataFrame(thresholds_list)

# Split
type_a = threshold_df[threshold_df['significant']].sort_values('review_moat')
type_b = threshold_df[~threshold_df['significant']].sort_values('review_moat')

# Create figure with two subplots
fig, (ax_a, ax_b) = plt.subplots(1, 2, figsize=(16, 7), gridspec_kw={'width_ratios': [7, 3]})

# ── Left panel: Type A — Review-Locked ──
colors_a = [CB_PALETTE[list(results.keys()).index(c)] for c in type_a['category']]
bars_a = ax_a.barh(type_a['category'], type_a['review_moat'], color=colors_a, edgecolor='white', linewidth=0.5)
for bar, val, sig in zip(bars_a, type_a['review_moat'], type_a['sig_label']):
    ax_a.text(bar.get_width() + 15000, bar.get_y() + bar.get_height()/2,
              f'{val:,.0f}  {sig}', va='center', fontweight='bold', fontsize=9)
ax_a.set_xlabel('Review Count Threshold (P(top-20) = 50%)')
ax_a.set_title('Type A: "Review-Locked" Categories\n(Reviews significantly predict rank)', fontsize=12, fontweight='bold')
ax_a.set_xlim(0, max(type_a['review_moat']) * 1.25)

# ── Right panel: Type B — Review-Independent ──
colors_b = [CB_PALETTE[list(results.keys()).index(c)] for c in type_b['category']]
bars_b = ax_b.barh(type_b['category'], [1]*len(type_b), color=colors_b, edgecolor='white', linewidth=0.5, alpha=0.4)
# Add "N/A" labels
for i, (bar, row) in enumerate(zip(bars_b, type_b.itertuples())):
    ax_b.text(0.5, bar.get_y() + bar.get_height()/2,
              f'N/A — no significant correlation\n(ρ={row.spearman_rho:.3f}, p={row.spearman_p:.3f})',
              ha='center', va='center', fontweight='bold', fontsize=8, color='#555555')
ax_b.set_xlim(0, 1)
ax_b.set_xticks([])
ax_b.set_title('Type B: "Review-Independent"\n(No meaningful threshold)', fontsize=12, fontweight='bold')
ax_b.spines['top'].set_visible(False)
ax_b.spines['right'].set_visible(False)
ax_b.spines['bottom'].set_visible(False)

fig.suptitle('The "Review Moat": Reviews Needed for 50% Probability of Top-20 Rank',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
fig.savefig(f"{VIZ_DIR}/{DATE}_chart4_review_moat_bars.png", bbox_inches='tight', dpi=150)
plt.close()
print("    [OK] Saved — Type A/B split")



  Chart 4: Review Moat Bars (Type A/B split)


    [OK] Saved — Type A/B split


In [20]:
# --- Chart 5: Category Accessibility Index ---
print("  Chart 5: Category Accessibility Index")
fig, ax = plt.subplots(figsize=(12, 6))
colors_acc = [CB_PALETTE[list(results.keys()).index(c)] for c in acc_df['category']]
bars = ax.barh(acc_df['category'], acc_df['accessibility_index'], color=colors_acc, edgecolor='white', linewidth=0.5)
for bar, val in zip(bars, acc_df['accessibility_index']):
    ax.text(bar.get_width() + 0.5, bar.get_y() + bar.get_height()/2, f'{val:.1f}', va='center', fontweight='bold', fontsize=10)
ax.set_xlabel('Accessibility Index (0-100, higher = easier to enter)')
ax.set_title('Category Accessibility Index\n(Review barrier 40% · Price dispersion 30% · Brand concentration 30%)', fontsize=13, fontweight='bold')
ax.set_xlim(0, 105)
plt.tight_layout()
fig.savefig(f"{VIZ_DIR}/{DATE}_chart5_accessibility_index.png", bbox_inches='tight', dpi=150)
plt.close()
print("    [OK] Saved")



  Chart 5: Category Accessibility Index
    [OK] Saved


In [21]:
# --- Chart 6: Price × Reviews Interaction Heatmap ---
print("  Chart 6: Price × Reviews Interaction")
# Bin price and reviews
df['price_bin'] = pd.qcut(df['price_usd'], q=5, labels=['Very Low', 'Low', 'Medium', 'High', 'Very High'])
df['review_bin'] = pd.qcut(df['review_count'], q=5, labels=['Very Low', 'Low', 'Medium', 'High', 'Very High'])
heatmap_data = df.pivot_table(values='in_top20', index='review_bin', columns='price_bin', aggfunc='mean')
fig, ax = plt.subplots(figsize=(10, 7))
sns.heatmap(heatmap_data, annot=True, fmt='.0%', cmap='YlOrRd', ax=ax, vmin=0, vmax=0.5, cbar_kws={'label': 'P(Top 20)'})
ax.set_title('Probability of Top-20 Rank: Price × Reviews Interaction\n(All categories combined)', fontsize=13, fontweight='bold')
ax.set_xlabel('Price Tier')
ax.set_ylabel('Review Count Tier')
plt.tight_layout()
fig.savefig(f"{VIZ_DIR}/{DATE}_chart6_price_review_interaction.png", bbox_inches='tight', dpi=150)
plt.close()
print("    [OK] Saved")



  Chart 6: Price × Reviews Interaction


    [OK] Saved


In [22]:
# --- Chart 7: Rating vs Reviews — Which Matters More? ---
print("  Chart 7: Rating vs Reviews Importance")
importance_data = []
for cat in sorted(df['category'].unique()):
    cat_df = df[df['category'] == cat]
    r_reviews = abs(spearmanr(cat_df['review_count'], cat_df['bsr_rank'])[0])
    r_rating = abs(spearmanr(cat_df['rating'], cat_df['bsr_rank'])[0])
    importance_data.append({'category': cat, 'reviews_rho': r_reviews, 'rating_rho': r_rating, 'difference': r_reviews - r_rating})

imp_df = pd.DataFrame(importance_data).sort_values('difference', ascending=True)
fig, ax = plt.subplots(figsize=(12, 7))
x = np.arange(len(imp_df))
width = 0.35
bars1 = ax.barh(x - width/2, imp_df['reviews_rho'], width, label='Review Count |ρ|', color='#0072B2')
bars2 = ax.barh(x + width/2, imp_df['rating_rho'], width, label='Rating |ρ|', color='#E69F00')
ax.set_yticks(x)
ax.set_yticklabels(imp_df['category'])
ax.set_xlabel("Absolute Spearman's ρ with BSR Rank")
ax.set_title('Which Matters More for Rank: Reviews or Rating?\n(Higher bar = stronger relationship)', fontsize=13, fontweight='bold')
ax.legend()
ax.set_xlim(0, 0.5)
plt.tight_layout()
fig.savefig(f"{VIZ_DIR}/{DATE}_chart7_rating_vs_reviews_importance.png", bbox_inches='tight', dpi=150)
plt.close()
print("    [OK] Saved")



  Chart 7: Rating vs Reviews Importance
    [OK] Saved


In [23]:
# --- Chart 8: Review Count Distribution by Category (Violin) ---
print("  Chart 8: Review Distribution Violin Plot")
fig, ax = plt.subplots(figsize=(14, 7))
cat_order = sorted(df['category'].unique())
# Trim to 95th percentile for visibility
p95 = df['review_count'].quantile(0.95)
violin_data = [df[df['category']==c]['review_count'].clip(upper=p95).values for c in cat_order]
parts = ax.violinplot(violin_data, positions=range(len(cat_order)), showmeans=True, showmedians=True)
for i, pc in enumerate(parts['bodies']):
    pc.set_facecolor(CB_PALETTE[i])
    pc.set_alpha(0.7)
ax.set_xticks(range(len(cat_order)))
ax.set_xticklabels(cat_order, rotation=45, ha='right', fontsize=9)
ax.set_ylabel('Review Count')
ax.set_title(f'Review Count Distribution by Category\n(Trimmed at 95th percentile = {p95:,.0f} reviews)', fontsize=13, fontweight='bold')
plt.tight_layout()
fig.savefig(f"{VIZ_DIR}/{DATE}_chart8_review_distribution_violin.png", bbox_inches='tight', dpi=150)
plt.close()
print("    [OK] Saved")

print("\n[OK] All 8 visualizations saved.")



  Chart 8: Review Distribution Violin Plot


    [OK] Saved

[OK] All 8 visualizations saved.


In [24]:
# ── SURPRISE FINDINGS ──────────────────────────────────


In [25]:
print("\n" + "=" * 70)
print("5. KEY FINDINGS & SURPRISES")



5. KEY FINDINGS & SURPRISES


In [26]:
print("=" * 70)

# Finding 1: Categories with lowest review barriers
print("\n[FIND] FINDING 1: Lowest Review Barriers")
low_barrier = cat_summary.sort_values('pct_under_50', ascending=False)
print(low_barrier[['pct_under_50', 'pct_under_100', 'median_reviews']].head(5).to_string())

# Finding 2: Review moat thresholds (using threshold_df from Chart 4)
print("\n[FIND] FINDING 2: Review Moat Thresholds")
t_summary = threshold_df.set_index('category')
print(t_summary[['review_moat', 'spearman_rho', 'spearman_p', 'type']].to_string())

# Finding 3: Rating vs Reviews
print("\n[FIND] FINDING 3: Where Rating Matters More Than Reviews")
rating_wins = imp_df[imp_df['difference'] < 0]
if len(rating_wins) > 0:
    print(f"Categories where RATING is more predictive than reviews: {rating_wins['category'].tolist()}")
else:
    # Find where the gap is smallest
    closest = imp_df.nsmallest(3, 'difference')
    print(f"Categories where rating almost matters as much: {closest['category'].tolist()}")

# Finding 4: Price×Review interaction
print("\n[FIND] FINDING 4: Price-Review Interaction")
print(heatmap_data.to_string())

# Finding 5: Overall
# How many categories have 30%+ of top 100 with under 50 reviews
low_barrier_cats = cat_summary[cat_summary['pct_under_50'] >= 30]
print(f"\n[FIND] FINDING 5: {len(low_barrier_cats)}/{len(cat_summary)} categories have 30%+ of top-100 products with <50 reviews")
print(low_barrier_cats[['pct_under_50', 'pct_under_100']].to_string())

# Overall stats
print(f"\n[STAT] OVERALL: Across all 10 categories:")
print(f"  % of top-100 with <50 reviews: {cat_summary['pct_under_50'].mean():.1f}%")
print(f"  % of top-100 with <100 reviews: {cat_summary['pct_under_100'].mean():.1f}%")
# Use Type A thresholds only for meaningful median
type_a_thresholds = threshold_df[threshold_df['significant']]['review_moat']
print(f"  Median review moat threshold (Type A only): {type_a_thresholds.median():,.0f}")
print(f"  Easiest category: {acc_df.iloc[0]['category']} (Index={acc_df.iloc[0]['accessibility_index']:.1f})")
print(f"  Hardest category: {acc_df.iloc[-1]['category']} (Index={acc_df.iloc[-1]['accessibility_index']:.1f})")
print(f"  Type A (Review-Locked) categories: {len(type_a_thresholds)}/10")
print(f"  Type B (Review-Independent) categories: {len(threshold_df) - len(type_a_thresholds)}/10")

print("\n[OK] Analysis complete!")



[FIND] FINDING 1: Lowest Review Barriers
                    pct_under_50  pct_under_100  median_reviews
category                                                       
Electronics                  3.0            3.0         19015.0
Toys & Games                 3.0            6.0          8117.5
Kitchen & Dining             1.0            1.0         19800.0
Health & Household           1.0            1.0         57990.0
Baby                         0.0            0.0         25506.5

[FIND] FINDING 2: Review Moat Thresholds
                           review_moat  spearman_rho  spearman_p                        type
category                                                                                    
Baby                      1.101878e+05     -0.445017    0.000004       Type A: Review-Locked
Electronics               0.000000e+00     -0.179669    0.075160  Type B: Review-Independent
Health & Household        7.487743e+05     -0.296670    0.002725       Type A: Review-Locked
Hom

## Conclusions

See `../tasks/active/review-moat-analysis/analysis.md` for the full written analysis with content drafts.

### Key Takeaways
- The Review Moat is real: only 0.8% of top-100 products have less than 50 reviews
- But 3/10 categories are Review-Independent (Electronics, Sports and Outdoors, Toys and Games)
- Toys and Games is the most accessible category (Index = 73.9/100)
- In Toys and Games, rating matters MORE than review count for predicting rank
- Baby is the most impenetrable category (Index = 0.2/100)
